In [3]:
import os

for root, dirs, files in os.walk("/"):
    for file in files:
        if file.endswith(".csv"):
            print(os.path.join(root, file))

/usr/lib/R/site-library/gargle/discovery-doc-ingest/parameter-properties.csv
/usr/lib/R/site-library/gargle/discovery-doc-ingest/api-wide-parameters.csv
/usr/lib/R/site-library/gargle/discovery-doc-ingest/method-properties.csv
/usr/lib/R/site-library/readr/extdata/mini-gapminder-americas.csv
/usr/lib/R/site-library/readr/extdata/mtcars.csv
/usr/lib/R/site-library/readr/extdata/mini-gapminder-oceania.csv
/usr/lib/R/site-library/readr/extdata/mini-gapminder-africa.csv
/usr/lib/R/site-library/readr/extdata/challenge.csv
/usr/lib/R/site-library/readr/extdata/mini-gapminder-asia.csv
/usr/lib/R/site-library/readr/extdata/chickens.csv
/usr/lib/R/site-library/readr/extdata/mini-gapminder-europe.csv
/usr/lib/R/site-library/lifecycle/lintr/linters.csv
/usr/lib/R/site-library/data.table/tests/doublequote_newline.csv
/usr/lib/R/site-library/data.table/tests/russellCRCRLF.csv
/usr/lib/R/site-library/data.table/tests/quoted_no_header.csv
/usr/lib/R/site-library/data.table/tests/russellCRLF.csv
/usr/

In [10]:
train = pd.read_csv(
    "/content/drive/MyDrive/IV_Surface/data/raw/dataset.csv"
)

print(train.shape)

(975, 30)


In [4]:
import os

print(os.listdir("/content"))

['.config', 'sample_data']


In [6]:
import os

print(os.listdir("/content"))

['.config', 'sample_data']


In [7]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [8]:
import os

for root, dirs, files in os.walk('/content/drive'):
    for f in files:
        if 'dataset' in f.lower():
            print(os.path.join(root, f))

/content/drive/MyDrive/IV_Surface/data/raw/dataset.csv


In [9]:
import os
print(os.listdir("/content"))

['.config', 'drive', 'sample_data']


In [11]:
option_cols = train.columns[2:].tolist()

print("Option columns:", len(option_cols))

Option columns: 28


In [12]:
def get_missing_gaps(mask):

    gaps = []
    start = None

    for i, val in enumerate(mask):

        if val and start is None:
            start = i

        elif not val and start is not None:
            gaps.append((start, i - 1, i - start))
            start = None

    if start is not None:
        gaps.append((start, len(mask) - 1, len(mask) - start))

    return gaps

In [13]:
def gap_validation(df, option_cols, random_state=42):

    rng = np.random.default_rng(random_state)

    errors = {
        1: [],
        2: [],
        3: [],
        4: []
    }

    for row_idx in range(len(df)):

        row = df.loc[row_idx, option_cols].values.astype(float)

        known_idx = np.where(~np.isnan(row))[0]

        if len(known_idx) < 10:
            continue

        hide_n = max(1, int(len(known_idx) * 0.20))

        hide_idx = rng.choice(
            known_idx,
            size=hide_n,
            replace=False
        )

        original = row.copy()

        row[hide_idx] = np.nan

        x = np.arange(len(row))

        valid = ~np.isnan(row)

        if valid.sum() < 2:
            continue

        f = interp1d(
            x[valid],
            row[valid],
            kind="linear",
            fill_value="extrapolate"
        )

        pred = f(x)

        hidden_mask = np.zeros(len(row), dtype=bool)
        hidden_mask[hide_idx] = True

        gaps = get_missing_gaps(hidden_mask)

        for start, end, gap_size in gaps:

            true_vals = original[start:end+1]
            pred_vals = pred[start:end+1]

            mse = np.mean(
                (true_vals - pred_vals) ** 2
            )

            bucket = gap_size if gap_size <= 3 else 4

            errors[bucket].append(mse)

    return {
        k: np.mean(v)
        for k, v in errors.items()
    }

In [14]:
results = gap_validation(
    train,
    option_cols,
    random_state=42
)

print(results)

{1: np.float64(0.00018770330437575035), 2: np.float64(0.00019147898801775114), 3: np.float64(0.00023311128029355284), 4: np.float64(7.223685000000042e-05)}
